# Feature selection

Two stages. **Stage 1 — null importance (all-relevant):** keep a feature when its real GBM gain beats its **own** 95th-percentile shuffled-target null — a per-feature noise floor, lenient enough to hold the long tail of weak-but-real signal. **Stage 2 — backward stability (redundancy trim):** randomized backward elimination over the survivors; keep anything that survived ≥ 1 of 8 runs, so only the *provably-redundant* (a twin always covers them) are cut — no signal lost. Chosen null over Boruta (see `llm-collaboration/30-selection-null-vs-boruta`).

In [1]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
from pathlib import Path
import lightgbm as lgb
from src.data import load_master
from src.select import null_importance, backward_stability

INTERIM = Path("../data/interim")

def gbm():
    return lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31,
                              min_child_samples=50, subsample=0.8, colsample_bytree=0.7,
                              n_jobs=-1, verbose=-1)

In [2]:
X, y = load_master()
X.shape

(307511, 3570)

## Stage 1 — null importance (all-relevant)

In [3]:
res = null_importance(gbm, X, y, k=20, n=80_000)
relevant = res.index[res["keep"]].tolist()
print(f"{len(relevant)} relevant / {len(res)}")
res.head(30)

null run  1/20
null run  2/20
null run  3/20
null run  4/20
null run  5/20
null run  6/20
null run  7/20
null run  8/20
null run  9/20
null run 10/20
null run 11/20
null run 12/20
null run 13/20
null run 14/20
null run 15/20
null run 16/20
null run 17/20
null run 18/20
null run 19/20
null run 20/20
442 relevant / 3570


,actual,null_mean,null_95,null_max,keep
ext_calc_mean,23085.046381,226.509706,321.754969,323.826329,True
ext_calc_median,8827.393884,220.483463,343.323966,393.716240,True
ext_calc_min,2428.167815,272.592398,398.528481,405.632640,True
ext_calc_2x3,2263.399091,332.216783,412.584351,424.280043,True
ext_calc_max,2019.568177,282.995924,370.077745,384.649207,True
credit_markup,1750.787738,221.481939,287.466685,304.381891,True
days_employed_percentage,1508.313257,356.804223,447.678169,451.918149,True
prev_DAYS_LAST_DUE_1ST_VERSION_max,1316.193415,192.896210,287.591107,301.464221,True
x_pca_7,1299.003480,162.617978,243.027408,283.547688,True
DAYS_EMPLOYED,1214.247474,315.891475,493.393670,515.707249,True


## Stage 2 — backward stability (redundancy trim)

Max recall: keep anything that survived ≥ 1 of 8 backward runs; only features droppable in *every* ordering (the provably-redundant) are cut.

In [4]:
# stage-2 model is lighter (fewer trees, single-threaded) and the runs go in parallel
def fast_gbm():
    return lgb.LGBMClassifier(n_estimators=150, learning_rate=0.05, num_leaves=31,
                              min_child_samples=50, subsample=0.8, colsample_bytree=0.7,
                              n_jobs=1, verbose=-1)

freq = backward_stability(fast_gbm, X[relevant], y, k_runs=8, n=30_000, batch=25, n_jobs=-1)
freq.head(30)

  run 2 | batch   1 | kept  417 | logloss 0.2427
  run 5 | batch   1 | kept  417 | logloss 0.2333
  run 6 | batch   1 | kept  438 | logloss 0.2371
  run 5 | batch   2 | kept  416 | logloss 0.2322
  run 7 | batch   1 | kept  436 | logloss 0.2436
  run 2 | batch   2 | kept  415 | logloss 0.2413
  run 1 | batch   1 | kept  437 | logloss 0.2407
  run 3 | batch   1 | kept  434 | logloss 0.2415
  run 4 | batch   1 | kept  441 | logloss 0.2447
  run 0 | batch   1 | kept  442 | logloss 0.2419
  run 3 | batch   2 | kept  409 | logloss 0.2411
  run 5 | batch   3 | kept  416 | logloss 0.2322
  run 6 | batch   2 | kept  437 | logloss 0.2367
  run 2 | batch   3 | kept  413 | logloss 0.2410
  run 7 | batch   2 | kept  434 | logloss 0.2434
  run 3 | batch   3 | kept  409 | logloss 0.2411
  run 1 | batch   2 | kept  436 | logloss 0.2404
  run 4 | batch   2 | kept  440 | logloss 0.2439
  run 0 | batch   2 | kept  442 | logloss 0.2419
  run 5 | batch   4 | kept  416 | logloss 0.2322
  run 2 | batch   4 

ext_calc_mean                                            1.0
bureau_AMT_CREDIT_SUM_OVERDUE_std                        1.0
x_cc_active_AMT_CREDIT_LIMIT_ACTUAL_sum_to_credit        1.0
bureau_bb_MONTHS_BALANCE_max_mean                        1.0
bureau_CREDIT_TYPE_Car_loan_mean                         1.0
OCCUPATION_TYPE_Core_staff                               1.0
prev_ref_AMT_DOWN_PAYMENT_min                            1.0
bureau_closed_credit_ongoing_mean                        1.0
bureau_bb_STATUS_C_sum_min                               1.0
bureau_overdue_1825                                      1.0
cc_active_cash_share_sum                                 1.0
bureau_AMT_CREDIT_MAX_OVERDUE_sum                        1.0
bureau_bb_dpd_skew_60_mean                               1.0
bureau_active_bb_STATUS_X_sum_min                        1.0
cc_util_mean_12                                          1.0
cc_min_pay_ratio_60                                      1.0
prev_ref_application_cre

## Keep

In [5]:
THRESH = 0.1  # survived >= 1 of 8 runs (any unique signal kept)
kept = freq[freq >= THRESH].index.tolist()
dropped = freq[freq < THRESH].index.tolist()
print(f"{len(kept)} kept / {len(relevant)} relevant  ({len(dropped)} always-redundant dropped)")
kept[:30]

441 kept / 442 relevant  (1 always-redundant dropped)


['ext_calc_mean',
 'bureau_AMT_CREDIT_SUM_OVERDUE_std',
 'x_cc_active_AMT_CREDIT_LIMIT_ACTUAL_sum_to_credit',
 'bureau_bb_MONTHS_BALANCE_max_mean',
 'bureau_CREDIT_TYPE_Car_loan_mean',
 'OCCUPATION_TYPE_Core_staff',
 'prev_ref_AMT_DOWN_PAYMENT_min',
 'bureau_closed_credit_ongoing_mean',
 'bureau_bb_STATUS_C_sum_min',
 'bureau_overdue_1825',
 'cc_active_cash_share_sum',
 'bureau_AMT_CREDIT_MAX_OVERDUE_sum',
 'bureau_bb_dpd_skew_60_mean',
 'bureau_active_bb_STATUS_X_sum_min',
 'cc_util_mean_12',
 'cc_min_pay_ratio_60',
 'prev_ref_application_credit_ratio_min',
 'x_prev_ref_AMT_CREDIT_sum_to_income',
 'bureau_CREDIT_TYPE_Microloan_mean',
 'x_bureau_closed_AMT_CREDIT_MAX_OVERDUE_mean_to_credit',
 'DEF_60_CNT_SOCIAL_CIRCLE',
 'prev_NAME_PAYMENT_TYPE_Cash_through_the_bank_sum',
 'prev_NFLAG_LAST_APPL_IN_DAY_sum',
 'bureau_active_AMT_CREDIT_SUM_OVERDUE_max',
 'cnt_non_child',
 'bureau_closed_CREDIT_TYPE_Consumer_credit_sum',
 'bureau_active_count',
 'prev_ref_NAME_YIELD_GROUP_middle_mean',
 '

# Save

In [6]:
pd.Series(kept, name="feature").to_csv(INTERIM / "selected_features.csv", index=False)
len(kept)

441